# 09 · Multivariate (Joint) Probabilistic Forecasts

So far every forecast was **univariate**: one series, one horizon at a time. Many decisions depend on several
quantities **jointly**:

* total demand across stores sharing a warehouse (the *sum* of correlated series)
* a portfolio of assets (VaR of the sum)
* electricity load at several nodes, wind farms in one region
* the same series over several horizons (path dependence, nb 05)

A multivariate forecast is a joint distribution $\hat F(y_1,\dots,y_d)$ = **marginals + dependence**.

## Theory

### Sklar's theorem
Every joint CDF can be written as $F(y_1,\dots,y_d)=C\big(F_1(y_1),\dots,F_d(y_d)\big)$, where $C$ is a **copula**
(a joint CDF on $[0,1]^d$ with uniform marginals). So you can model the marginals one by one (with anything:
ARIMA, quantile regression, GBM…) and then **glue them together** with a copula:

| Approach | Dependence model |
|---|---|
| Independence | $C(u)=\prod u_i$, usually wrong |
| Gaussian copula | correlation matrix $R$ of $\Phi^{-1}(u)$ |
| t-copula | adds tail dependence (crashes happen together) |
| Vine copulas | flexible pair-wise building blocks |
| **Empirical copula / Schaake shuffle** | reorder marginal samples using the rank structure of a historical template |
| **Ensemble copula coupling (ECC)** | reuse the rank structure of a raw dynamic ensemble |
| Full joint models | VAR / VECM, multivariate state-space, DeepVAR, GPVar, multivariate diffusion |

### Why you cannot ignore dependence
For $S=\sum_iY_i$: $\text{Var}(S)=\sum_i\sigma_i^2+2\sum_{i<j}\rho_{ij}\sigma_i\sigma_j$. With positive correlation,
assuming independence makes intervals for the total **too narrow**.

### Multivariate scores
* **Energy score** $\text{ES}=\mathbb E\lVert\mathbf X-\mathbf y\rVert-\tfrac12\mathbb E\lVert\mathbf X-\mathbf X'\rVert$: proper, generalises CRPS, but
  only *weakly* sensitive to the correlation structure. Dominated by the large-scale components.
* **Variogram score** (Scheuerer & Hamill 2015) $\text{VS}_p=\sum_{i<j}w_{ij}\big(|y_i-y_j|^p-\mathbb E|X_i-X_j|^p\big)^2$:
  compares pairwise differences, so it is **sensitive to dependence**. ($p=0.5$ is the usual choice.)
* **Multivariate log score** (needs a density), **Dawid–Sebastiani** score (needs mean and covariance).
* CRPS of the **aggregate** (e.g. total sales) is a practical check of whether the dependence is right.

Important: **marginal scores (CRPS per series) are identical** for all methods that share the same marginals.
Only joint scores can tell them apart.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(9)

## 1. Marginal forecasts for the four stores
Per-store SARIMA on log-sales (parameters from 2015–2019), one-step forecasts for 2020–2024 (60 months).
We also keep the **in-sample standardised one-step residuals** (2016–2019) to learn the cross-store dependence.

In [2]:
retail = pf.make_retail_hierarchy()
S_names = pf.BOTTOM_NAMES
CUT = 60
MU, SD, Y, Z = [], [], [], []
for s in S_names:
    ly = np.log(retail[s])
    fit = SARIMAX(ly.iloc[:CUT], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    r = SARIMAX(ly, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).filter(fit.params)
    pr = r.get_prediction(start=13)
    m, v = np.asarray(pr.predicted_mean), np.sqrt(np.asarray(pr.var_pred_mean))
    obs = ly.iloc[13:].to_numpy()
    k = CUT - 13
    Z.append(((obs - m) / v)[:k])            # training-period standardised residuals
    MU.append(m[k:]); SD.append(v[k:]); Y.append(retail[s].iloc[CUT:].to_numpy())
MU, SD, Y, Z = (np.column_stack(a) for a in (MU, SD, Y, Z))   # shapes (T, 4) / (47, 4)
R = np.corrcoef(Z.T)
test_idx = retail.index[CUT:]
pd.DataFrame(R, index=S_names, columns=S_names).round(2)

,North_A,North_B,South_A,South_B
North_A,1.00,0.76,0.24,0.22
North_B,0.76,1.00,0.30,0.28
South_A,0.24,0.30,1.00,0.62
South_B,0.22,0.28,0.62,1.00


The residual correlation is strong, and strongest within a region (North_A–North_B, South_A–South_B).
That is the common + regional shock structure of the data generator.

## 2. Four joint forecasters, all with **identical marginals**
* **Independent**: $Z\sim N(0,I)$
* **Gaussian copula**: $Z\sim N(0,\hat R)$ with $\hat R$ from training residuals
* **Schaake shuffle**: independent marginal samples, re-ordered to follow the ranks of historical residual vectors
* **Too correlated**: $\rho=0.95$ everywhere (a mis-specified copula)

Joint samples on the log scale are $\mu+\sigma\odot Z$, then exponentiated.

In [3]:
M = 500
def gaussian_copula_samples(Rm):
    L = np.linalg.cholesky(Rm)
    z = rng.standard_normal((len(MU), M, 4)) @ L.T
    return np.exp(MU[:, None, :] + SD[:, None, :] * z)          # (T, M, 4)

def schaake_samples():
    indep = np.exp(MU[:, None, :] + SD[:, None, :] * rng.standard_normal((len(MU), M, 4)))
    out = np.empty_like(indep)
    for t in range(len(MU)):
        template = Z[rng.integers(0, len(Z), M)] + 1e-9 * rng.standard_normal((M, 4))   # bootstrap template rows
        ranks = np.argsort(np.argsort(template, axis=0), axis=0)
        srt = np.sort(indep[t], axis=0)
        out[t] = np.take_along_axis(srt, ranks, axis=0)          # same marginal values, new pairing
    return out

R_hi = np.full((4, 4), 0.95); np.fill_diagonal(R_hi, 1)
J = {"Independent": gaussian_copula_samples(np.eye(4)),
     "Gaussian copula (R̂)": gaussian_copula_samples(R),
     "Schaake shuffle": schaake_samples(),
     "Too correlated (ρ=.95)": gaussian_copula_samples(R_hi)}

In [4]:
t0 = 30
fig = pf.subplots(1, 4, titles=list(J), shared_xaxes=True, shared_yaxes=True, height=420,
                  title=f"Joint predictive samples for {test_idx[t0]:%b %Y}: same marginals, different dependence")
for c, (name, smp) in enumerate(J.items(), start=1):
    fig.add_trace(go.Scatter(x=smp[t0, :, 0], y=smp[t0, :, 1], mode="markers", name="joint samples", legendgroup="s",
                             showlegend=c == 1, marker=dict(color=pf.SERIES[0], size=5, opacity=0.35),
                             hovertemplate="North_A %{x:,.0f}, North_B %{y:,.0f}<extra></extra>"), row=1, col=c)
    fig.add_trace(go.Scatter(x=[Y[t0, 0]], y=[Y[t0, 1]], mode="markers", name="actual", legendgroup="a", showlegend=c == 1,
                             marker=dict(color=pf.ACTUAL, size=16, symbol="star", line=dict(color=pf.INK["surface"], width=2)),
                             hovertemplate="actual: North_A %{x:,.0f}, North_B %{y:,.0f}<extra></extra>"), row=1, col=c)
fig.update_xaxes(title_text="North_A")
fig.update_yaxes(title_text="North_B", row=1, col=1)
fig.update_layout(margin=dict(t=100))
fig.show()

The Schaake-shuffle cloud looks *clumpy*: its pairing comes from only 47 historical residual vectors (re-sampled), so
samples cluster around those observed dependence patterns. With a longer history (or ECC on a dynamic ensemble)
the empirical copula becomes smoother.

## 3. Scores: marginal vs joint vs aggregate

In [5]:
rows = {}
total_y = Y.sum(axis=1)
for name, smp in J.items():
    crps_marg = np.mean([pf.crps_sample(Y[:, i], smp[:, :, i].T).mean() for i in range(4)])
    es = np.mean([pf.energy_score(Y[t], smp[t]) for t in range(len(Y))])
    vs = np.mean([pf.variogram_score(Y[t], smp[t], p=0.5) for t in range(len(Y))])
    tot = smp.sum(axis=2)                                       # (T, M) samples of the TOTAL
    lo, hi = np.quantile(tot, 0.05, axis=1), np.quantile(tot, 0.95, axis=1)
    rows[name] = {"mean CRPS (per store)": crps_marg, "energy score": es, "variogram score": vs,
                  "CRPS of total": pf.crps_sample(total_y, tot.T).mean(),
                  "90% coverage of total": pf.coverage(total_y, lo, hi), "90% width of total": np.mean(hi - lo)}
pd.DataFrame(rows).T

,mean CRPS (per store),energy score,variogram score,CRPS of total,90% coverage of total,90% width of total
Independent,21.172,49.935,18.156,64.958,0.800,271.678
Gaussian copula (R̂),21.222,49.296,17.657,64.410,0.900,399.091
Schaake shuffle,21.287,49.681,17.872,65.457,0.850,374.352
Too correlated (ρ=.95),21.289,50.713,19.116,66.354,0.967,517.472


In [6]:
fig = pf.subplots(1, 4, titles=[f"PIT of TOTAL: {name}" for name in J], shared_yaxes=True, height=340)
for c, smp in enumerate(J.values(), start=1):
    pf.add_pit(fig, pf.pit_from_samples(total_y, smp.sum(axis=2).T), bins=6, row=1, col=c)
fig.show()

**Reading the results**
* *Mean CRPS per store* is the same up to Monte-Carlo noise, as it must be, because the marginals are identical.
* The **variogram score** ranks the dependence models correctly (estimated copula best, ρ=.95 worst). The differences
  in the **energy score** are small relative to its size. It is known to have weak sensitivity to correlation, so don't
  rely on it alone.
* The clearest signal is the **aggregate**: the independent forecaster's 90% intervals for the *total* are about 30%
  narrower and noticeably under-cover. The "too correlated" one is far too wide. The estimated copula and the Schaake
  shuffle are in between and close to nominal.
* With 60 test months these differences are real but noisy. In practice, evaluate joint forecasts over long backtests.

## 4. Takeaways
* Model marginals with your best univariate method, then add dependence with a copula. The empirical copula
  (Schaake shuffle) needs no parametric assumption.
* Evaluate dependence with the **variogram score** and with **aggregate** checks. Marginal scores cannot see it.
* The same tools apply to dependence **across horizons** (paths) and across **levels of a hierarchy** (nb 10).

### Exercises
1. Replace the Gaussian copula with a t-copula (ν=4). Does the score of the total improve in months with large shocks?
2. Estimate $\hat R$ from only the last 24 months (rolling). How noisy does the estimate get?
3. Compute the Dawid–Sebastiani score $\log\det\Sigma+(\mathbf y-\mu)^\top\Sigma^{-1}(\mathbf y-\mu)$ for each forecaster (on the log scale).